In [3]:
import os
import asyncio
from typing import Any
from mcp.server.fastmcp import FastMCP
from langchain.agents import create_agent 
from langchain_openai import ChatOpenAI
from langchain_mcp_adapters.client import MultiServerMCPClient
from dotenv import load_dotenv

load_dotenv()

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [ ]:
from ast import Mult
from http import client


model = ChatOpenAI(
		model="gpt-4.1-mini",
		api_key=os.getenv("OPENAI_API_KEY"),
		base_url=os.getenv("OPENAI_API_BASE"),
		temperature=0
	)

client = MultiServerMCPClient(
	{
		"calculator": {
			"command": "python",
			"args": ["/root/code/mcp_servers/calculator.py"],
			"transport": "stdio"
		},
		
		"weather": {
			"command": "python",
			"args": ["/root/code/mcp_servers/weather_server.py"],
			"transport": "stdio"
			
		}
	}
)

async def run_agent_with_mcp():
	"""Create and run agent with multiple MCP servers"""
	tools = await client.get_tools()
	print(f"Loaded {len(tools) if hasattr(tools, '__len__') else 'multiple'} tools from MCP servers")

	agent = create_agent(model, tools)

	print("TESTING MULTI-SERVER ORCHESTRATION")
	print("=" * 60)

	print("\nTest 1: Calculator MCP")
	calc_response = await agent.ainvoke({
        "messages": [{"role": "user", "content": "What is 42 plus 58?"}]
    })
	print(f"Response: {calc_response['messages'][-1].content}")

	print("\nTest 2: Weather MCP")
	weather_response = await agent.ainvoke({
		"messages": [{"role": "user", "content": "Compare weather in fNew York and Tokydo"}]
	})
	print(f"Reponse: {weather_response['messages'][-1].content}")
									 

	print("\nTest 3")
	mixed_response = await agent.ainvoke({
		"messages": [{"role": "user", "content":"If it is 20 degree celcius in Paris and temperature rises by 5 degrees celcius, what will it be"}]
	})

	print(f"Respnse: {mixed_response['messages'][-1].content}")
	
await run_agent_with_mcp()

FileNotFoundError: [Errno 2] No such file or directory: 'python'